
1. Verify the Cleaned Dataset Path

**Objective:** Confirm the location and accessibility of the cleaned GDELT dataset.

Verify that the directory exists, identify the Parquet files, and check their sizes before beginning feature engineering.

No files will be modified.


In [2]:

# Update the cleaned dataset path
CLEANED_DIR = (
    PROJECT_DIR
    / "gdelt_data"
    / "processed"
    / "gdelt_cleaned_by_date_v2"
)

# Verify directory
print("Directory exists:", CLEANED_DIR.exists())
print("Full path:", CLEANED_DIR.resolve())

# Verify Parquet files
if CLEANED_DIR.is_dir():
    parquet_files = sorted(CLEANED_DIR.rglob("*.parquet"))

    print("\nParquet files found:", len(parquet_files))

    total_size = 0

    for file in parquet_files:
        size_mb = file.stat().st_size / (1024**2)
        total_size += size_mb

        print(f"{file.relative_to(CLEANED_DIR)} — {size_mb:.2f} MiB")

    print(f"\nTotal dataset size: {total_size / 1024:.2f} GiB")
else:
    print("ERROR: Cleaned dataset directory not found.")


Directory exists: True
Full path: C:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\gdelt_data\processed\gdelt_cleaned_by_date_v2

Parquet files found: 6
DateAddedYear=2025\DateAddedMonth=01\part-0001.parquet — 261.32 MiB
DateAddedYear=2025\DateAddedMonth=02\part-0001.parquet — 248.09 MiB
DateAddedYear=2025\DateAddedMonth=03\part-0001.parquet — 273.12 MiB
DateAddedYear=2025\DateAddedMonth=04\part-0001.parquet — 265.40 MiB
DateAddedYear=2025\DateAddedMonth=05\part-0001.parquet — 271.64 MiB
DateAddedYear=2025\DateAddedMonth=06\part-0001.parquet — 112.90 MiB

Total dataset size: 1.40 GiB



### 2. Inspect Cleaned Dataset Schema

**Objective:** Verify the structure and columns of the cleaned GDELT dataset.

Inspect the Parquet schema, confirm the number of records and columns, and identify the fields required for geopolitical feature engineering.

Only Parquet metadata will be read, avoiding loading the entire dataset into memory.


In [3]:

# Read schema and row counts from Parquet metadata
first_file = pq.ParquetFile(parquet_files[0])

schema = first_file.schema_arrow

total_rows = sum(
    pq.ParquetFile(file).metadata.num_rows
    for file in parquet_files
)

print("Total records:", f"{total_rows:,}")
print("Total columns:", len(schema.names))

print("\nColumn names and data types:")
for i, field in enumerate(schema, start=1):
    print(f"{i:2}. {field.name:35} {field.type}")


Total records: 20,343,168
Total columns: 90

Column names and data types:
 1. GLOBALEVENTID                       int64
 2. SQLDATE                             int64
 3. MonthYear                           int64
 4. Year                                int64
 5. FractionDate                        double
 6. Actor1Code                          large_string
 7. Actor1Name                          large_string
 8. Actor1CountryCode                   large_string
 9. Actor1KnownGroupCode                large_string
10. Actor1EthnicCode                    large_string
11. Actor1Religion1Code                 large_string
12. Actor1Religion2Code                 large_string
13. Actor1Type1Code                     large_string
14. Actor1Type2Code                     large_string
15. Actor1Type3Code                     large_string
16. Actor2Code                          large_string
17. Actor2Name                          large_string
18. Actor2CountryCode                   large_string
19. Ac


### 3. Define Oil-Exporting and Oil-Importing Countries

**Objective:** Establish country classifications for geopolitical feature engineering.

Following Vodenska et al. (2016), we select 10 major crude oil exporters and 10 major crude oil importers.

These classifications will be used to calculate country-level geopolitical indicators and directional relationships between exporters and importers.

Worldwide events will remain included in the global indicators. Countries outside these groups will not be discarded from the cleaned dataset.

The classifications are historical and will be reviewed against net crude oil trade before finalizing directional features.


In [4]:

# Major crude oil exporters from Vodenska et al. (2016)
OIL_EXPORTERS = {
    "SA": "Saudi Arabia",
    "RS": "Russia",
    "KU": "Kuwait",
    "IR": "Iran",
    "IZ": "Iraq",
    "NI": "Nigeria",
    "AE": "United Arab Emirates",
    "AO": "Angola",
    "VE": "Venezuela",
    "NO": "Norway"
}

# Major crude oil importers from Vodenska et al. (2016)
OIL_IMPORTERS = {
    "US": "United States",
    "CH": "China",
    "IN": "India",
    "JA": "Japan",
    "KS": "South Korea",
    "GM": "Germany",
    "IT": "Italy",
    "FR": "France",
    "NL": "Netherlands",
    "SN": "Singapore"
}

# Countries included in our oil-trade analysis
OIL_COUNTRIES = set(OIL_EXPORTERS) | set(OIL_IMPORTERS)

print("Exporting countries:", len(OIL_EXPORTERS))
print("Importing countries:", len(OIL_IMPORTERS))
print("Total selected countries:", len(OIL_COUNTRIES))

print("\nCountries appearing in both groups:",
      set(OIL_EXPORTERS) & set(OIL_IMPORTERS))


Exporting countries: 10
Importing countries: 10
Total selected countries: 20

Countries appearing in both groups: set()



### 4. Define CAMEO Event Categories

**Objective:** Establish the 20 standard CAMEO EventRootCode categories for geopolitical feature engineering.

Following our research approach, we retain all categories of international activity rather than focusing only on conflicts.

These categories will be used to calculate daily event counts worldwide and for selected oil-exporting and oil-importing countries.

The 16 previously identified malformed event classifications will not contribute to CAMEO category counts, but their records will remain in the cleaned dataset and general event totals.


In [5]:

# Standard CAMEO root event categories
CAMEO_ROOTS = {
    "01": "Make public statement",
    "02": "Appeal",
    "03": "Express intent to cooperate",
    "04": "Consult",
    "05": "Engage in diplomatic cooperation",
    "06": "Engage in material cooperation",
    "07": "Provide aid",
    "08": "Yield",
    "09": "Investigate",
    "10": "Demand",
    "11": "Disapprove",
    "12": "Reject",
    "13": "Threaten",
    "14": "Protest",
    "15": "Exhibit military posture",
    "16": "Reduce relations",
    "17": "Coerce",
    "18": "Assault",
    "19": "Fight",
    "20": "Use unconventional mass violence"
}

# Confirm the required classifications
assert len(CAMEO_ROOTS) == 20
assert len(set(CAMEO_ROOTS)) == 20

print("CAMEO root categories:", len(CAMEO_ROOTS))

for code, description in CAMEO_ROOTS.items():
    print(f"{code} — {description}")


CAMEO root categories: 20
01 — Make public statement
02 — Appeal
03 — Express intent to cooperate
04 — Consult
05 — Engage in diplomatic cooperation
06 — Engage in material cooperation
07 — Provide aid
08 — Yield
09 — Investigate
10 — Demand
11 — Disapprove
12 — Reject
13 — Threaten
14 — Protest
15 — Exhibit military posture
16 — Reduce relations
17 — Coerce
18 — Assault
19 — Fight
20 — Use unconventional mass violence



### 5. Define the Daily Feature Structure

**Objective:** Define the 84 daily geopolitical indicators before processing the cleaned GDELT dataset.

Features are organized into nine groups covering global events, CAMEO categories, oil-exporting and importing countries, directed international relationships, Goldstein scores, news tone, media attention, and data quality.

The feature list follows the methodology of Vodenska et al. (2016), supplemented with additional indicators for exploratory analysis.

This step establishes the expected output structure without loading or modifying event records.


In [6]:

FEATURE_GROUPS = {
    "general": [
        "date",
        "global_total_events",
        "exporter_total_events",
        "importer_total_events",
        "oil_country_event_share",
    ],

    "global_cameo": [
        f"global_root_{code}"
        for code in CAMEO_ROOTS
    ],

    "oil_country_cameo": [
        f"exporter_root_{code}"
        for code in [
            "01", "04", "05", "06", "10", "11",
            "13", "14", "15", "16", "17", "19"
        ]
    ] + [
        f"importer_root_{code}"
        for code in [
            "04", "05", "06", "11",
            "13", "14", "17", "19"
        ]
    ],

    "directed_relationships": [
        f"{relationship}_{metric}"
        for relationship in [
            "exp_to_exp",
            "exp_to_imp",
            "imp_to_exp",
            "imp_to_imp"
        ]
        for metric in [
            "events",
            "goldstein_mean",
            "tone_mean"
        ]
    ],

    "goldstein": [
        "global_goldstein_mean",
        "exporter_goldstein_mean",
        "importer_goldstein_mean",
        "global_goldstein_std",
        "exporter_goldstein_impact_index",
        "importer_goldstein_impact_index",
    ],

    "tone": [
        "global_tone_mean",
        "exporter_tone_mean",
        "importer_tone_mean",
        "global_negative_tone_share",
    ],

    "media_attention": [
        "global_total_mentions",
        "global_total_articles",
        "exporter_total_mentions",
        "exporter_total_articles",
        "global_mentions_per_event",
    ],

    "internal_and_quality": [
        "exporter_internal_event_share",
        "importer_internal_event_share",
        "actor1_country_missing_share",
        "actor2_country_missing_share",
    ],

    "major_countries": [
        "iran_total_events",
        "saudi_total_events",
        "russia_total_events",
        "iraq_total_events",
        "usa_total_events",
        "china_total_events",
        "iran_goldstein_mean",
        "russia_goldstein_mean",
    ],
}

FEATURE_COLUMNS = [
    feature
    for features in FEATURE_GROUPS.values()
    for feature in features
]

print("Feature groups:", len(FEATURE_GROUPS))
print("Total columns:", len(FEATURE_COLUMNS))
print("Duplicate names:", len(FEATURE_COLUMNS) - len(set(FEATURE_COLUMNS)))

for group, features in FEATURE_GROUPS.items():
    print(f"{group:25} {len(features):2} columns")

assert len(FEATURE_COLUMNS) == 84
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS))


Feature groups: 9
Total columns: 84
Duplicate names: 0
general                    5 columns
global_cameo              20 columns
oil_country_cameo         20 columns
directed_relationships    12 columns
goldstein                  6 columns
tone                       4 columns
media_attention            5 columns
internal_and_quality       4 columns
major_countries            8 columns



### 6. Define Country Membership and Directed Relationships

**Objective:** Establish consistent rules for assigning GDELT events to oil-exporting countries, oil-importing countries, and directed international relationships.

An event involving at least one selected exporter contributes once to exporter-level indicators; the same principle applies to importers. Worldwide indicators include all events.

Directed relationships use Actor1 as the initiating country and Actor2 as the receiving country. Both actor countries must be identified and belong to our selected groups.

The four directed classifications are mutually exclusive. Events with missing or unclassified actor countries remain available for global analysis.

Country membership is based on the historical groups defined in Step 3.


In [7]:

# Define membership sets
EXPORTER_CODES = set(OIL_EXPORTERS)
IMPORTER_CODES = set(OIL_IMPORTERS)

# Mutually exclusive directional classifications
RELATIONSHIP_TYPES = {
    ("exporter", "exporter"): "exp_to_exp",
    ("exporter", "importer"): "exp_to_imp",
    ("importer", "exporter"): "imp_to_exp",
    ("importer", "importer"): "imp_to_imp",
}

def classify_country(code):
    """Classify a country using the selected oil-trade groups."""
    if code in EXPORTER_CODES:
        return "exporter"
    if code in IMPORTER_CODES:
        return "importer"
    return None


def classify_relationship(actor1_code, actor2_code):
    """Return directed relationship category or None."""
    actor1_group = classify_country(actor1_code)
    actor2_group = classify_country(actor2_code)

    return RELATIONSHIP_TYPES.get(
        (actor1_group, actor2_group)
    )


# Test classification rules using known countries
examples = [
    ("SA", "RS"),  # Saudi Arabia -> Russia
    ("SA", "US"),  # Saudi Arabia -> United States
    ("US", "IR"),  # United States -> Iran
    ("US", "CH"),  # United States -> China
    ("IR", None),  # Missing Actor2
    ("BR", "SA"),  # Brazil outside selected groups
]

for actor1, actor2 in examples:
    result = classify_relationship(actor1, actor2)
    print(f"{str(actor1):4} -> {str(actor2):4} : {result}")


SA   -> RS   : exp_to_exp
SA   -> US   : exp_to_imp
US   -> IR   : imp_to_exp
US   -> CH   : imp_to_imp
IR   -> None : None
BR   -> SA   : None



### 7. Define Feature Aggregation Rules

**Objective:** Establish consistent calculation methods for the 84 daily geopolitical features.

Each event will be grouped by its DATEADDED date to reflect when the information entered GDELT.

Feature calculations will use the following methods:

- **Counts:** Number of qualifying event records.
- **Shares:** Qualifying events divided by the relevant total.
- **Means:** Average GoldsteinScale_Adjusted or AvgTone.
- **Standard deviation:** Variation in Goldstein scores.
- **Goldstein Impact Index:** Goldstein scores weighted by NumMentions, following Vodenska et al. (2016).
- **Media attention:** Sum of NumMentions and NumArticles.
- **Internal events:** Events where Actor1CountryCode and Actor2CountryCode match and are nonmissing.

Country-level features include events involving a selected country as either Actor1 or Actor2, counted once per event.

Directed relationship features preserve Actor1 → Actor2 order.

Undefined means and shares will remain missing rather than being incorrectly assigned zero.


In [8]:

# Define aggregation rules for the daily feature dataset
AGGREGATION_RULES = {
    "date_column": "DATEADDED",
    "event_id": "GLOBALEVENTID",
    "event_category": "EventRootCode",
    "goldstein_column": "GoldsteinScale_Adjusted",
    "tone_column": "AvgTone",
    "mentions_column": "NumMentions",
    "articles_column": "NumArticles",
    "actor1_country": "Actor1CountryCode",
    "actor2_country": "Actor2CountryCode",
}

# Original Goldstein Impact Index definition from Paper 1:
# sum(GoldsteinScale_Adjusted * NumMentions) / event_count
def goldstein_impact_index(goldstein, mentions):
    valid = goldstein.notna() & mentions.notna()

    if valid.sum() == 0:
        return np.nan

    return (
        (goldstein[valid] * mentions[valid]).sum()
        / valid.sum()
    )


# Safe division for shares and ratios
def safe_ratio(numerator, denominator):
    if denominator == 0 or pd.isna(denominator):
        return np.nan
    return numerator / denominator


print("Aggregation rules defined:", len(AGGREGATION_RULES))
print("Goldstein Impact Index function: ready")
print("Safe ratio function: ready")


Aggregation rules defined: 9
Goldstein Impact Index function: ready
Safe ratio function: ready



### 8. Load a Sample for Feature Engineering

**Objective:** Load a small sample of cleaned GDELT records to test feature-engineering calculations.

We will read 10,000 records from the first Parquet file, selecting only the columns required for our analysis.

This allows us to test country classifications, event categories, numerical indicators, and date handling without loading the full dataset.

The original cleaned Parquet files will remain unchanged.


In [9]:

# Columns required for feature engineering
REQUIRED_COLUMNS = [
    "GLOBALEVENTID",
    "DATEADDED",
    "Actor1CountryCode",
    "Actor2CountryCode",
    "EventRootCode",
    "GoldsteinScale_Adjusted",
    "AvgTone",
    "NumMentions",
    "NumArticles",
]

# Read only the required columns and first 10,000 records
sample = next(
    pq.ParquetFile(parquet_files[0]).iter_batches(
        batch_size=10_000,
        columns=REQUIRED_COLUMNS
    )
).to_pandas()

print("Sample shape:", sample.shape)
print("\nData types:")
print(sample.dtypes)

print("\nFirst 5 records:")
display(sample.head())


Sample shape: (10000, 9)

Data types:
GLOBALEVENTID                int64
DATEADDED                    int64
Actor1CountryCode           string
Actor2CountryCode           string
EventRootCode               string
GoldsteinScale_Adjusted    float64
AvgTone                    float64
NumMentions                  int64
NumArticles                  int64
dtype: object

First 5 records:


,GLOBALEVENTID,DATEADDED,Actor1CountryCode,Actor2CountryCode,EventRootCode,GoldsteinScale_Adjusted,AvgTone,NumMentions,NumArticles
0,1218366341,20250101,<NA>,USA,11,-2.0,-5.281690,2,2
1,1218366342,20250101,<NA>,USA,18,-8.0,-5.281690,2,2
2,1218366343,20250101,CAN,<NA>,01,0.4,-0.607952,100,100
3,1218366344,20250101,USA,<NA>,01,0.4,-6.726224,10,10
4,1218366345,20250101,USA,<NA>,04,1.0,1.385281,6,6



### 9. Verify Actor Country Code Format

**Objective:** Check compatibility between GDELT actor-country codes and our exporter/importer classification dictionaries.

The cleaned dataset contains three-letter actor-country codes, while our initial country lists use two-letter codes from the research paper.

We will inspect the distinct country codes in the sample to determine the correct format before calculating country-level or directed relationship features.

No data will be modified.

Note: this is a cleaning check.

In [10]:

# Inspect distinct actor-country codes in the sample
for column in ["Actor1CountryCode", "Actor2CountryCode"]:
    codes = sample[column].dropna()

    print(f"\n{column}")
    print("Code lengths:")
    print(codes.str.len().value_counts().sort_index())

    print("\nMost frequent codes:")
    print(codes.value_counts().head(15))

# Check whether our existing country codes match the data
sample_codes = (
    set(sample["Actor1CountryCode"].dropna())
    | set(sample["Actor2CountryCode"].dropna())
)

print("\nMatching exporter codes:", sorted(EXPORTER_CODES & sample_codes))
print("Matching importer codes:", sorted(IMPORTER_CODES & sample_codes))



Actor1CountryCode
Code lengths:
Actor1CountryCode
3    5396
Name: count, dtype: Int64

Most frequent codes:
Actor1CountryCode
USA    2217
GBR     208
CHN     175
ISR     175
AUS     172
NGA     162
RUS     162
CAN     125
UKR     114
PAK      94
PSE      79
FRA      70
SYR      68
IRN      61
CUB      55
Name: count, dtype: int64[pyarrow]

Actor2CountryCode
Code lengths:
Actor2CountryCode
3    3986
Name: count, dtype: Int64

Most frequent codes:
Actor2CountryCode
USA    1403
RUS     174
CHN     152
ISR     145
GBR     137
NGA     134
UKR     127
AUS     100
CAN      96
PSE      95
SYR      75
FRA      75
PAK      69
IRN      46
JPN      42
Name: count, dtype: int64[pyarrow]

Matching exporter codes: []
Matching importer codes: []



### 10. Correct Oil-Country Classification Codes

**Objective:** Correct the country-code mismatch discovered during feature engineering.

The cleaned GDELT dataset uses three-letter CAMEO country codes for Actor1CountryCode and Actor2CountryCode. Our initial exporter/importer dictionaries incorrectly used two-letter codes.

We replace the classification dictionaries with valid three-letter CAMEO codes and verify their compatibility with the sample data.

This correction affects the reference mappings only. The cleaned Parquet dataset remains unchanged.

**Design decision:** Document the mismatch as a post-cleaning compatibility issue and preserve the original country codes.


In [11]:

# Correct three-letter CAMEO country codes

OIL_EXPORTERS = {
    "SAU": "Saudi Arabia",
    "RUS": "Russia",
    "KWT": "Kuwait",
    "IRN": "Iran",
    "IRQ": "Iraq",
    "NGA": "Nigeria",
    "ARE": "United Arab Emirates",
    "AGO": "Angola",
    "VEN": "Venezuela",
    "NOR": "Norway"
}

OIL_IMPORTERS = {
    "USA": "United States",
    "CHN": "China",
    "IND": "India",
    "JPN": "Japan",
    "KOR": "South Korea",
    "DEU": "Germany",
    "ITA": "Italy",
    "FRA": "France",
    "NLD": "Netherlands",
    "SGP": "Singapore"
}

# Update membership sets used by classification functions
EXPORTER_CODES = set(OIL_EXPORTERS)
IMPORTER_CODES = set(OIL_IMPORTERS)
OIL_COUNTRIES = EXPORTER_CODES | IMPORTER_CODES

# Check country mappings against sample data
sample_codes = (
    set(sample["Actor1CountryCode"].dropna())
    | set(sample["Actor2CountryCode"].dropna())
)

print("Exporters:", len(EXPORTER_CODES))
print("Importers:", len(IMPORTER_CODES))
print("Overlapping countries:", EXPORTER_CODES & IMPORTER_CODES)

print("\nExporter codes found in sample:",
      sorted(EXPORTER_CODES & sample_codes))

print("Importer codes found in sample:",
      sorted(IMPORTER_CODES & sample_codes))

print("\nDirectional classification checks:")
for a1, a2 in [
    ("SAU", "RUS"),
    ("SAU", "USA"),
    ("USA", "IRN"),
    ("USA", "CHN")
]:
    print(f"{a1} -> {a2}: {classify_relationship(a1, a2)}")


Exporters: 10
Importers: 10
Overlapping countries: set()

Exporter codes found in sample: ['AGO', 'ARE', 'IRN', 'IRQ', 'KWT', 'NGA', 'NOR', 'RUS', 'SAU', 'VEN']
Importer codes found in sample: ['CHN', 'DEU', 'FRA', 'IND', 'ITA', 'JPN', 'KOR', 'NLD', 'SGP', 'USA']

Directional classification checks:
SAU -> RUS: exp_to_exp
SAU -> USA: exp_to_imp
USA -> IRN: imp_to_exp
USA -> CHN: imp_to_imp



### 11. Test Country Membership and Relationship Classification

**Objective:** Apply the corrected oil-country classifications to 10,000 cleaned GDELT records.

For each event, identify whether it involves an oil exporter, an oil importer, or both. Classify directed relationships using Actor1 → Actor2.

Each event is counted only once within each country group. Events with unidentified actors remain available for global analysis.

This test verifies our classification logic before processing the complete dataset.


In [12]:

# Work on a copy to preserve the original sample
test_df = sample.copy()

# Identify events involving selected oil countries
test_df["is_exporter_event"] = (
    test_df["Actor1CountryCode"].isin(EXPORTER_CODES)
    | test_df["Actor2CountryCode"].isin(EXPORTER_CODES)
)

test_df["is_importer_event"] = (
    test_df["Actor1CountryCode"].isin(IMPORTER_CODES)
    | test_df["Actor2CountryCode"].isin(IMPORTER_CODES)
)

test_df["is_oil_country_event"] = (
    test_df["is_exporter_event"]
    | test_df["is_importer_event"]
)

# Classify directed country relationships
test_df["relationship_type"] = [
    classify_relationship(a1, a2)
    for a1, a2 in zip(
        test_df["Actor1CountryCode"],
        test_df["Actor2CountryCode"]
    )
]

print("Total sample events:", len(test_df))
print("Exporter events:", test_df["is_exporter_event"].sum())
print("Importer events:", test_df["is_importer_event"].sum())
print("Oil-country events:", test_df["is_oil_country_event"].sum())

print("\nDirected relationship counts:")
print(
    test_df["relationship_type"]
    .value_counts(dropna=False)
)


Total sample events: 10000
Exporter events: 764
Importer events: 4107
Oil-country events: 4794

Directed relationship counts:
relationship_type
NaN           9429
imp_to_imp     406
exp_to_exp      88
imp_to_exp      45
exp_to_imp      32
Name: count, dtype: int64



### 12. Test Daily Event Count Aggregation

**Objective:** Calculate the first daily geopolitical features using the 10,000-record sample.

Aggregate events by DATEADDED to produce daily totals for worldwide activity, oil-exporting countries, oil-importing countries, and the four directed relationship categories.

Calculate the proportion of worldwide events involving our selected oil countries.

This is a preliminary test of aggregation logic, not the final daily dataset.


In [13]:

# Convert DATEADDED into calendar dates
test_df["date"] = pd.to_datetime(
    test_df["DATEADDED"].astype(str),
    format="%Y%m%d"
)

# Aggregate event activity by date
daily_test = test_df.groupby("date").agg(
    global_total_events=("GLOBALEVENTID", "size"),
    exporter_total_events=("is_exporter_event", "sum"),
    importer_total_events=("is_importer_event", "sum"),
    oil_country_events=("is_oil_country_event", "sum")
)

# Calculate oil-country reporting share
daily_test["oil_country_event_share"] = (
    daily_test["oil_country_events"]
    / daily_test["global_total_events"]
)

# Aggregate directed relationship counts
relationship_counts = pd.crosstab(
    test_df["date"],
    test_df["relationship_type"]
)

for relationship in [
    "exp_to_exp",
    "exp_to_imp",
    "imp_to_exp",
    "imp_to_imp"
]:
    daily_test[f"{relationship}_events"] = (
        relationship_counts[relationship]
        if relationship in relationship_counts.columns
        else 0
    )

daily_test = daily_test.fillna({
    f"{r}_events": 0
    for r in ["exp_to_exp", "exp_to_imp", "imp_to_exp", "imp_to_imp"]
})

print("Daily observations:", len(daily_test))
display(daily_test.head())

print("\nCount validation:")
print("Global events:", daily_test["global_total_events"].sum())
print("Oil-country events:", daily_test["oil_country_events"].sum())
print(
    "Directed relationships:",
    daily_test[[f"{r}_events" for r in RELATIONSHIP_TYPES.values()]]
    .sum().sum()
)


Daily observations: 1


,global_total_events,exporter_total_events,importer_total_events,oil_country_events,oil_country_event_share,exp_to_exp_events,exp_to_imp_events,imp_to_exp_events,imp_to_imp_events
date,,,,,,,,,
2025-01-01,10000,764,4107,4794,0.4794,88,32,45,406



Count validation:
Global events: 10000
Oil-country events: 4794
Directed relationships: 571



13. Calculate Daily CAMEO Event Counts

**Objective:** Calculate daily event counts by CAMEO root category for worldwide events and selected oil-exporting and importing countries.

All 20 CAMEO root categories are retained for worldwide analysis. Following our 84-feature specification, 12 selected categories are calculated for exporters and 8 for importers.

Events involving both actors in the same country group are counted only once within that group.

This test uses 10,000 cleaned records to validate the calculations before full-dataset processing.


In [14]:

# Identify the CAMEO features specified in Cell 5
GLOBAL_ROOT_FEATURES = [
    f for f in FEATURE_GROUPS["global_cameo"]
]

EXPORTER_ROOT_FEATURES = [
    f for f in FEATURE_GROUPS["oil_country_cameo"]
    if f.startswith("exporter_root_")
]

IMPORTER_ROOT_FEATURES = [
    f for f in FEATURE_GROUPS["oil_country_cameo"]
    if f.startswith("importer_root_")
]

# Count events by date and root category
def calculate_root_counts(df, mask, prefix, feature_names):
    counts = pd.crosstab(
        df.loc[mask, "date"],
        df.loc[mask, "EventRootCode"]
    )

    result = pd.DataFrame(index=daily_test.index)

    for feature in feature_names:
        code = feature.split("_")[-1]

        result[feature] = (
            counts[code].reindex(result.index, fill_value=0)
            if code in counts.columns
            else 0
        )

    return result


global_counts = calculate_root_counts(
    test_df,
    pd.Series(True, index=test_df.index),
    "global",
    GLOBAL_ROOT_FEATURES
)

exporter_counts = calculate_root_counts(
    test_df,
    test_df["is_exporter_event"],
    "exporter",
    EXPORTER_ROOT_FEATURES
)

importer_counts = calculate_root_counts(
    test_df,
    test_df["is_importer_event"],
    "importer",
    IMPORTER_ROOT_FEATURES
)

# Add CAMEO features to the daily test dataset
daily_test = pd.concat(
    [daily_test, global_counts, exporter_counts, importer_counts],
    axis=1
)

print("Global CAMEO features:", len(GLOBAL_ROOT_FEATURES))
print("Exporter CAMEO features:", len(EXPORTER_ROOT_FEATURES))
print("Importer CAMEO features:", len(IMPORTER_ROOT_FEATURES))
print("Total CAMEO features:", len(GLOBAL_ROOT_FEATURES)
      + len(EXPORTER_ROOT_FEATURES)
      + len(IMPORTER_ROOT_FEATURES))

print("\nGlobal CAMEO events counted:",
      daily_test[GLOBAL_ROOT_FEATURES].sum().sum())

print("\nTop 10 global event categories:")
display(
    daily_test[GLOBAL_ROOT_FEATURES]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)


Global CAMEO features: 20
Exporter CAMEO features: 12
Importer CAMEO features: 8
Total CAMEO features: 40

Global CAMEO events counted: 10000

Top 10 global event categories:


global_root_04    2338
global_root_01    1326
global_root_05     952
global_root_19     857
global_root_02     720
global_root_11     653
global_root_17     636
global_root_03     502
global_root_08     370
global_root_07     353
dtype: int64


### 14. Calculate Goldstein and News Tone Indicators

**Objective:** Calculate daily geopolitical intensity and sentiment indicators from GDELT event records.

Following Vodenska et al. (2016), we calculate average Goldstein scores, Goldstein variability, the Goldstein Impact Index, and average news tone.

Indicators are calculated worldwide, for oil exporters and importers, and for the four directed relationship categories.

We use the adjusted Goldstein scores from the cleaning stage and preserve missing values where a group has no valid observations.

This test uses the 10,000-record sample before processing the full dataset.


In [15]:

# Group definitions
GROUP_MASKS = {
    "global": pd.Series(True, index=test_df.index),
    "exporter": test_df["is_exporter_event"],
    "importer": test_df["is_importer_event"],
}

RELATIONSHIPS = [
    "exp_to_exp",
    "exp_to_imp",
    "imp_to_exp",
    "imp_to_imp",
]

# Calculate indicators
for group, mask in GROUP_MASKS.items():
    subset = test_df.loc[mask]

    # Mean Goldstein score
    daily_test[f"{group}_goldstein_mean"] = (
        subset.groupby("date")["GoldsteinScale_Adjusted"]
        .mean()
        .reindex(daily_test.index)
    )

    # Mean news tone
    daily_test[f"{group}_tone_mean"] = (
        subset.groupby("date")["AvgTone"]
        .mean()
        .reindex(daily_test.index)
    )

# Global Goldstein standard deviation
daily_test["global_goldstein_std"] = (
    test_df.groupby("date")["GoldsteinScale_Adjusted"]
    .std()
    .reindex(daily_test.index)
)

# Goldstein Impact Index for exporters and importers
for group in ["exporter", "importer"]:
    subset = test_df.loc[GROUP_MASKS[group]]

    daily_test[f"{group}_goldstein_impact_index"] = (
        subset.groupby("date")
        .apply(
            lambda g: goldstein_impact_index(
                g["GoldsteinScale_Adjusted"],
                g["NumMentions"]
            ),
            include_groups=False
        )
        .reindex(daily_test.index)
    )

# Relationship-specific Goldstein and tone means
for relationship in RELATIONSHIPS:
    subset = test_df.loc[
        test_df["relationship_type"] == relationship
    ]

    daily_test[f"{relationship}_goldstein_mean"] = (
        subset.groupby("date")["GoldsteinScale_Adjusted"]
        .mean()
        .reindex(daily_test.index)
    )

    daily_test[f"{relationship}_tone_mean"] = (
        subset.groupby("date")["AvgTone"]
        .mean()
        .reindex(daily_test.index)
    )

# Share of globally reported events with negative tone
daily_test["global_negative_tone_share"] = (
    test_df.assign(negative_tone=test_df["AvgTone"] < 0)
    .groupby("date")["negative_tone"]
    .mean()
    .reindex(daily_test.index)
)

# Display results
INDICATORS = [
    "global_goldstein_mean",
    "exporter_goldstein_mean",
    "importer_goldstein_mean",
    "global_goldstein_std",
    "exporter_goldstein_impact_index",
    "importer_goldstein_impact_index",
    "global_tone_mean",
    "exporter_tone_mean",
    "importer_tone_mean",
    "global_negative_tone_share",
]

print("Indicators calculated:", len(INDICATORS))
print("\nGlobal, exporter and importer indicators:")
display(daily_test[INDICATORS].round(4))

print("\nDirected relationship indicators:")
display(
    daily_test[
        [f"{r}_{m}" for r in RELATIONSHIPS
         for m in ["goldstein_mean", "tone_mean"]]
    ].round(4)
)


Indicators calculated: 10

Global, exporter and importer indicators:


,global_goldstein_mean,exporter_goldstein_mean,importer_goldstein_mean,global_goldstein_std,exporter_goldstein_impact_index,importer_goldstein_impact_index,global_tone_mean,exporter_tone_mean,importer_tone_mean,global_negative_tone_share
date,,,,,,,,,,
2025-01-01,0.088,-0.5615,0.0429,4.8892,-39.9187,1.1206,-2.1537,-2.0829,-2.3195,0.6798



Directed relationship indicators:


,exp_to_exp_goldstein_mean,exp_to_exp_tone_mean,exp_to_imp_goldstein_mean,exp_to_imp_tone_mean,imp_to_exp_goldstein_mean,imp_to_exp_tone_mean,imp_to_imp_goldstein_mean,imp_to_imp_tone_mean
date,,,,,,,,
2025-01-01,-0.7398,-2.811,2.3938,-0.246,-0.48,-1.3584,-0.1419,-2.5114



### 15. Calculate Daily Media Attention Features

**Objective:** Measure daily news-reporting intensity worldwide and for major oil-exporting countries.

Following Vodenska et al. (2016), we aggregate NumMentions and NumArticles to measure the reporting intensity associated with geopolitical events.

We also calculate average mentions per event to distinguish changes in reporting intensity from changes in event volume.

These are event-associated reporting totals, not necessarily counts of unique news articles. Raw values are preserved; logarithmic transformations will be considered later during EDA.

This step uses the 10,000-record sample to validate calculations before full-dataset processing.


In [16]:

# Global daily reporting totals
global_media = test_df.groupby("date").agg(
    global_total_mentions=("NumMentions", "sum"),
    global_total_articles=("NumArticles", "sum")
)

# Reporting totals for events involving oil exporters
exporter_media = (
    test_df.loc[test_df["is_exporter_event"]]
    .groupby("date")
    .agg(
        exporter_total_mentions=("NumMentions", "sum"),
        exporter_total_articles=("NumArticles", "sum")
    )
)

# Add media features to the daily dataset
daily_test = daily_test.join(global_media)
daily_test = daily_test.join(exporter_media)

# Average event mentions
daily_test["global_mentions_per_event"] = (
    daily_test["global_total_mentions"]
    / daily_test["global_total_events"]
)

MEDIA_FEATURES = FEATURE_GROUPS["media_attention"]

print("Media attention features:", len(MEDIA_FEATURES))
display(daily_test[MEDIA_FEATURES].round(3))

# Basic checks
assert (
    daily_test["global_total_mentions"]
    >= daily_test["exporter_total_mentions"]
).all()

assert (
    daily_test["global_total_articles"]
    >= daily_test["exporter_total_articles"]
).all()

print("\nMedia attention checks passed.")


Media attention features: 5


,global_total_mentions,global_total_articles,exporter_total_mentions,exporter_total_articles,global_mentions_per_event
date,,,,,
2025-01-01,223682,214595,13474,13228,22.368



Media attention checks passed.



### 16. Calculate Internal Events and Geographic Data Quality

**Objective:** Calculate internal-event proportions and daily actor-country missingness indicators.

Following Vodenska et al. (2016), internal events are identified when Actor1 and Actor2 belong to the same country.

We calculate internal-event shares separately for oil exporters and importers. We also measure missing actor-country information across all events.

These indicators help distinguish domestic geopolitical activity from international interactions and assess whether missing geographic information could affect our analysis.

This step uses the 10,000-record sample before full-dataset processing.


In [17]:

# Identify events where both actors belong to the same country
test_df["is_internal_event"] = (
    test_df["Actor1CountryCode"].notna()
    & test_df["Actor2CountryCode"].notna()
    & (
        test_df["Actor1CountryCode"]
        == test_df["Actor2CountryCode"]
    )
).fillna(False)

# Internal-event shares for exporters and importers
for group in ["exporter", "importer"]:
    subset = test_df.loc[test_df[f"is_{group}_event"]]

    daily_test[f"{group}_internal_event_share"] = (
        subset.groupby("date")["is_internal_event"]
        .mean()
        .reindex(daily_test.index)
    )

# Daily actor-country missingness
daily_test["actor1_country_missing_share"] = (
    test_df["Actor1CountryCode"].isna()
    .groupby(test_df["date"])
    .mean()
    .reindex(daily_test.index)
)

daily_test["actor2_country_missing_share"] = (
    test_df["Actor2CountryCode"].isna()
    .groupby(test_df["date"])
    .mean()
    .reindex(daily_test.index)
)

QUALITY_FEATURES = FEATURE_GROUPS["internal_and_quality"]

print("Internal-event and quality features:", len(QUALITY_FEATURES))
display(daily_test[QUALITY_FEATURES].round(4))

# Validate proportions
assert daily_test[QUALITY_FEATURES].stack().between(0, 1).all()

print("\nAll proportions are within [0, 1].")


Internal-event and quality features: 4


,exporter_internal_event_share,importer_internal_event_share,actor1_country_missing_share,actor2_country_missing_share
date,,,,
2025-01-01,0.072,0.0704,0.4604,0.6014



All proportions are within [0, 1].



### 17. Calculate Major-Country Indicators

**Objective:** Calculate daily geopolitical activity for six strategically important countries and average Goldstein scores for Iran and Russia.

Country-level indicators supplement our global and exporter/importer features without replacing them.

An event involving a selected country as either Actor1 or Actor2 is counted once, even if both actors belong to that country.

We use GoldsteinScale_Adjusted from the cleaning stage. Missing Goldstein scores are excluded from means without removing the corresponding events.

This completes the calculation of our planned 84 features on the 10,000-record sample.


In [18]:

# Selected countries and their GDELT actor-country codes
MAJOR_COUNTRIES = {
    "iran": "IRN",
    "saudi": "SAU",
    "russia": "RUS",
    "iraq": "IRQ",
    "usa": "USA",
    "china": "CHN"
}

# Calculate country-level event counts
for country, code in MAJOR_COUNTRIES.items():

    country_mask = (
        test_df["Actor1CountryCode"].eq(code).fillna(False)
        | test_df["Actor2CountryCode"].eq(code).fillna(False)
    )

    country_events = test_df.loc[country_mask]

    daily_test[f"{country}_total_events"] = (
        country_events.groupby("date")
        .size()
        .reindex(daily_test.index, fill_value=0)
    )

    # Goldstein means for Iran and Russia
    if country in ["iran", "russia"]:
        daily_test[f"{country}_goldstein_mean"] = (
            country_events.groupby("date")["GoldsteinScale_Adjusted"]
            .mean()
            .reindex(daily_test.index)
        )

# Display country-level features
COUNTRY_FEATURES = FEATURE_GROUPS["major_countries"]

print("Major-country features calculated:", len(COUNTRY_FEATURES))

display(
    daily_test[COUNTRY_FEATURES].round(4)
)

# Verify country counts are within global event totals
COUNT_COLUMNS = [
    f"{country}_total_events"
    for country in MAJOR_COUNTRIES
]

assert all(
    (daily_test[column] <= daily_test["global_total_events"]).all()
    for column in COUNT_COLUMNS
)

print("\nMajor-country count checks passed.")


Major-country features calculated: 8


,iran_total_events,saudi_total_events,russia_total_events,iraq_total_events,usa_total_events,china_total_events,iran_goldstein_mean,russia_goldstein_mean
date,,,,,,,,
2025-01-01,107,30,321,21,3363,309,-2.2888,-1.7016



Major-country count checks passed.



### 18. Consolidate and Validate the Daily Feature Dataset

**Objective:** Combine the engineered indicators into the final 84-column structure and verify their completeness.

We will check that all planned features exist, confirm there are no duplicate columns, validate daily event counts, and identify any missing feature values.

This validation uses the 10,000-record sample. It does not recheck the cleaning work or modify the original Parquet dataset.

Once the feature structure is validated, the same aggregation logic can be applied to all 20.34 million GDELT records.


In [19]:

# Check that all planned features have been calculated
available_columns = set(daily_test.columns) | {"date"}

missing_features = [
    col for col in FEATURE_COLUMNS
    if col not in available_columns
]

print("Expected columns:", len(FEATURE_COLUMNS))
print("Missing features:", missing_features)

assert not missing_features, "Some planned features are missing."

# Convert date index into a regular column
daily_features_test = daily_test.reset_index()

# Select exactly the planned 84 columns, in specification order
daily_features_test = daily_features_test[FEATURE_COLUMNS].copy()

print("\nDataset shape:", daily_features_test.shape)
print("Duplicate columns:", daily_features_test.columns.duplicated().sum())
print("Duplicate dates:", daily_features_test["date"].duplicated().sum())

# Identify any missing indicator values
missing_values = daily_features_test.isna().sum()
missing_values = missing_values[missing_values > 0]

print("\nFeatures containing missing values:")
print(missing_values if not missing_values.empty else "None")

# Confirm sample event totals
assert daily_features_test["global_total_events"].sum() == len(test_df)
assert daily_features_test.shape[1] == 84

print("\nAll 84 feature columns validated.")

display(daily_features_test.iloc[:, :10].head())


Expected columns: 84
Missing features: []

Dataset shape: (1, 84)
Duplicate columns: 0
Duplicate dates: 0

Features containing missing values:
None

All 84 feature columns validated.


,date,global_total_events,exporter_total_events,importer_total_events,oil_country_event_share,global_root_01,global_root_02,global_root_03,global_root_04,global_root_05
0,2025-01-01,10000,764,4107,0.4794,1326,720,502,2338,952



### 19. Define the Scalable Batch Aggregation Function

**Objective:** Prepare the validated feature calculations for processing the complete GDELT dataset in memory-efficient batches.

The cleaned Parquet files will be read in batches rather than loaded entirely into memory.

For each batch, we calculate daily event counts, CAMEO categories, country activity, media-attention totals, and intermediate statistics required for Goldstein and AvgTone indicators.

We preserve sums, observation counts, and squared sums so that daily means, shares, standard deviations, and Goldstein Impact Indices can be calculated correctly after combining batches.

This step defines the processing function. It does not yet process the full dataset or modify the cleaned Parquet files.


In [20]:

# Columns needed from each Parquet batch
BATCH_COLUMNS = REQUIRED_COLUMNS

RELATIONSHIPS = [
    "exp_to_exp", "exp_to_imp",
    "imp_to_exp", "imp_to_imp"
]

def aggregate_feature_batch(df):
    """Return additive daily statistics for one GDELT batch."""

    dates = pd.to_datetime(
        df["DATEADDED"].astype(str), format="%Y%m%d"
    )

    a1 = df["Actor1CountryCode"]
    a2 = df["Actor2CountryCode"]
    root = df["EventRootCode"]

    gold = df["GoldsteinScale_Adjusted"].astype("float64")
    tone = df["AvgTone"].astype("float64")
    mentions = df["NumMentions"].astype("float64")
    articles = df["NumArticles"].astype("float64")

    exporter = (
        a1.isin(EXPORTER_CODES) | a2.isin(EXPORTER_CODES)
    ).fillna(False)

    importer = (
        a1.isin(IMPORTER_CODES) | a2.isin(IMPORTER_CODES)
    ).fillna(False)

    oil_country = exporter | importer

    internal = (
        a1.notna() & a2.notna() & a1.eq(a2)
    ).fillna(False)

    # Intermediate statistics are additive across batches
    data = {
        "global_total_events": np.ones(len(df), dtype="int64"),
        "exporter_total_events": exporter.astype("int64"),
        "importer_total_events": importer.astype("int64"),
        "_oil_country_events": oil_country.astype("int64"),
        "_global_gold_sum": gold.fillna(0),
        "_global_gold_count": gold.notna().astype("int64"),
        "_global_gold_sq_sum": gold.pow(2).fillna(0),
        "_exporter_gold_sum": gold.where(exporter, 0).fillna(0),
        "_exporter_gold_count": (exporter & gold.notna()).astype("int64"),
        "_importer_gold_sum": gold.where(importer, 0).fillna(0),
        "_importer_gold_count": (importer & gold.notna()).astype("int64"),
        "_global_tone_sum": tone.fillna(0),
        "_global_tone_count": tone.notna().astype("int64"),
        "_exporter_tone_sum": tone.where(exporter, 0).fillna(0),
        "_exporter_tone_count": (exporter & tone.notna()).astype("int64"),
        "_importer_tone_sum": tone.where(importer, 0).fillna(0),
        "_importer_tone_count": (importer & tone.notna()).astype("int64"),
        "_global_negative_tone_count": tone.lt(0).fillna(False).astype("int64"),
        "global_total_mentions": mentions,
        "global_total_articles": articles,
        "exporter_total_mentions": mentions.where(exporter, 0),
        "exporter_total_articles": articles.where(exporter, 0),
        "_exporter_internal_count": (exporter & internal).astype("int64"),
        "_importer_internal_count": (importer & internal).astype("int64"),
        "_actor1_missing_count": a1.isna().astype("int64"),
        "_actor2_missing_count": a2.isna().astype("int64"),
        "_exporter_gii_sum": (gold * mentions).where(exporter, 0).fillna(0),
        "_importer_gii_sum": (gold * mentions).where(importer, 0).fillna(0),
        "_exporter_gii_count": (
            exporter & gold.notna() & mentions.notna()
        ).astype("int64"),
        "_importer_gii_count": (
            importer & gold.notna() & mentions.notna()
        ).astype("int64"),
    }

    # Global CAMEO root counts
    for code in CAMEO_ROOTS:
        data[f"global_root_{code}"] = root.eq(code).fillna(False).astype("int64")

    # Selected exporter/importer root counts
    for feature in FEATURE_GROUPS["oil_country_cameo"]:
        group, _, code = feature.partition("_root_")
        mask = exporter if group == "exporter" else importer
        data[feature] = (mask & root.eq(code)).fillna(False).astype("int64")

    # Four mutually exclusive directed relationship categories
    a1_exp, a1_imp = a1.isin(EXPORTER_CODES), a1.isin(IMPORTER_CODES)
    a2_exp, a2_imp = a2.isin(EXPORTER_CODES), a2.isin(IMPORTER_CODES)

    relationship_masks = {
        "exp_to_exp": a1_exp & a2_exp,
        "exp_to_imp": a1_exp & a2_imp,
        "imp_to_exp": a1_imp & a2_exp,
        "imp_to_imp": a1_imp & a2_imp,
    }

    for name, mask in relationship_masks.items():
        mask = mask.fillna(False)
        data[f"{name}_events"] = mask.astype("int64")

        for metric, values in [("goldstein", gold), ("tone", tone)]:
            data[f"_{name}_{metric}_sum"] = values.where(mask, 0).fillna(0)
            data[f"_{name}_{metric}_count"] = (
                mask & values.notna()
            ).astype("int64")

    # Major-country features
    for name, code in MAJOR_COUNTRIES.items():
        mask = (a1.eq(code) | a2.eq(code)).fillna(False)
        data[f"{name}_total_events"] = mask.astype("int64")

        if name in ("iran", "russia"):
            data[f"_{name}_gold_sum"] = gold.where(mask, 0).fillna(0)
            data[f"_{name}_gold_count"] = (
                mask & gold.notna()
            ).astype("int64")

    # Aggregate intermediate statistics by reporting date
    batch_data = pd.DataFrame(data)
    batch_data.insert(0, "date", dates.to_numpy())

    return batch_data.groupby("date", sort=True).sum()


# Test the function against our existing 10,000-record sample
batch_test = aggregate_feature_batch(sample)

print("Batch aggregation function: ready")
print("Daily observations:", len(batch_test))
print("Intermediate statistics:", len(batch_test.columns))

print("\nSample validation:")
print("Global events:", int(batch_test["global_total_events"].sum()))
print("Oil-country events:", int(batch_test["_oil_country_events"].sum()))

directed_total = batch_test[
    [f"{r}_events" for r in RELATIONSHIPS]
].sum().sum()

print("Directed relationships:", int(directed_total))

assert batch_test["global_total_events"].sum() == 10_000
assert batch_test["_oil_country_events"].sum() == 4_794
assert directed_total == 571

print("\nBatch aggregation checks passed.")


Batch aggregation function: ready
Daily observations: 1
Intermediate statistics: 100

Sample validation:
Global events: 10000
Oil-country events: 4794
Directed relationships: 571

Batch aggregation checks passed.



### 20. Finalize Daily Feature Calculations

**Objective:** Convert intermediate batch statistics into the final 84 daily geopolitical features.

The aggregation function from Step 19 produces additive statistics such as counts, sums, and squared sums. We now calculate means, proportions, standard deviations, and Goldstein Impact Indices from these statistics.

This approach ensures accurate results when multiple processing batches are combined.

Undefined averages remain missing when no qualifying events exist. Event counts remain zero where no events were recorded.

We will validate the results against the independently calculated 10,000-record sample before processing the complete dataset.


In [21]:

def finalize_daily_features(stats):
    """Convert combined daily statistics into the final 84 features."""

    s = stats.copy()
    result = pd.DataFrame(index=s.index)

    def ratio(numerator, denominator):
        return numerator.div(denominator.replace(0, np.nan))

    # 1. General activity
    for col in [
        "global_total_events",
        "exporter_total_events",
        "importer_total_events",
    ]:
        result[col] = s[col]

    result["oil_country_event_share"] = ratio(
        s["_oil_country_events"], s["global_total_events"]
    )

    # 2. CAMEO category counts
    for group in ["global_cameo", "oil_country_cameo"]:
        for col in FEATURE_GROUPS[group]:
            result[col] = s[col]

    # 3. Directed relationship counts and averages
    for relationship in RELATIONSHIPS:
        result[f"{relationship}_events"] = s[f"{relationship}_events"]

        for metric in ["goldstein", "tone"]:
            result[f"{relationship}_{metric}_mean"] = ratio(
                s[f"_{relationship}_{metric}_sum"],
                s[f"_{relationship}_{metric}_count"]
            )

    # 4. Global/exporter/importer Goldstein means
    for group in ["global", "exporter", "importer"]:
        result[f"{group}_goldstein_mean"] = ratio(
            s[f"_{group}_gold_sum"],
            s[f"_{group}_gold_count"]
        )

    # Sample standard deviation (ddof=1)
    n = s["_global_gold_count"]
    variance = (
        s["_global_gold_sq_sum"]
        - s["_global_gold_sum"].pow(2).div(n.replace(0, np.nan))
    ) / (n - 1).replace(0, np.nan)

    result["global_goldstein_std"] = np.sqrt(variance.clip(lower=0))

    # Original Goldstein Impact Index formula from Paper 1
    for group in ["exporter", "importer"]:
        result[f"{group}_goldstein_impact_index"] = ratio(
            s[f"_{group}_gii_sum"],
            s[f"_{group}_gii_count"]
        )

    # 5. News tone
    for group in ["global", "exporter", "importer"]:
        result[f"{group}_tone_mean"] = ratio(
            s[f"_{group}_tone_sum"],
            s[f"_{group}_tone_count"]
        )

    result["global_negative_tone_share"] = ratio(
        s["_global_negative_tone_count"],
        s["global_total_events"]
    )

    # 6. Media attention
    for col in [
        "global_total_mentions",
        "global_total_articles",
        "exporter_total_mentions",
        "exporter_total_articles",
    ]:
        result[col] = s[col]

    result["global_mentions_per_event"] = ratio(
        s["global_total_mentions"],
        s["global_total_events"]
    )

    # 7. Internal events and missingness
    for group in ["exporter", "importer"]:
        result[f"{group}_internal_event_share"] = ratio(
            s[f"_{group}_internal_count"],
            s[f"{group}_total_events"]
        )

    for actor in ["actor1", "actor2"]:
        result[f"{actor}_country_missing_share"] = ratio(
            s[f"_{actor}_missing_count"],
            s["global_total_events"]
        )

    # 8. Major countries
    for country in MAJOR_COUNTRIES:
        result[f"{country}_total_events"] = s[f"{country}_total_events"]

        if country in ["iran", "russia"]:
            result[f"{country}_goldstein_mean"] = ratio(
                s[f"_{country}_gold_sum"],
                s[f"_{country}_gold_count"]
            )

    # Final 84-column structure
    result = result.reset_index()
    result = result[FEATURE_COLUMNS]

    return result


# Test finalization using our sample's intermediate statistics
finalized_test = finalize_daily_features(batch_test)

print("Final dataset shape:", finalized_test.shape)

# Compare against independently calculated sample features
pd.testing.assert_frame_equal(
    finalized_test,
    daily_features_test,
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

print("Independent sample comparison: PASSED")
print("All 84 feature calculations match.")

display(finalized_test.iloc[:, :10])


Final dataset shape: (1, 84)
Independent sample comparison: PASSED
All 84 feature calculations match.


,date,global_total_events,exporter_total_events,importer_total_events,oil_country_event_share,global_root_01,global_root_02,global_root_03,global_root_04,global_root_05
0,2025-01-01,10000,764,4107,0.4794,1326,720,502,2338,952


In [ ]:

### 21. Process the Complete GDELT Dataset

**Objective:** Generate daily geopolitical features from all 20,343,168 cleaned GDELT events.

We process the six monthly Parquet files in memory-efficient batches of 100,000 records, using the aggregation function validated in Steps 19–20.

Intermediate daily statistics are accumulated across batches and monthly partitions. After processing, these statistics are combined to calculate the final 84 geopolitical indicators.

The process preserves every event in the global totals, including records with missing actor-country information.

The original cleaned Parquet files remain unchanged.


In [22]:

import time

BATCH_SIZE = 100_000

daily_stats = None
processed_rows = 0
start_time = time.perf_counter()

print("Starting full GDELT feature aggregation...\n")

for file_number, file in enumerate(parquet_files, start=1):

    parquet = pq.ParquetFile(file)
    file_rows = 0

    for batch in parquet.iter_batches(
        batch_size=BATCH_SIZE,
        columns=BATCH_COLUMNS
    ):
        df = batch.to_pandas()

        # Calculate additive daily statistics
        batch_stats = aggregate_feature_batch(df)

        # Combine statistics across batches and files
        if daily_stats is None:
            daily_stats = batch_stats
        else:
            daily_stats = daily_stats.add(
                batch_stats,
                fill_value=0
            )

        file_rows += len(df)
        processed_rows += len(df)

        del df, batch_stats

    elapsed = time.perf_counter() - start_time

    print(
        f"[{file_number}/{len(parquet_files)}] "
        f"{file.parent.name}: "
        f"{file_rows:,} rows | "
        f"Total: {processed_rows:,} | "
        f"Elapsed: {elapsed:.1f}s",
        flush=True
    )

# Finalize all 84 features
daily_features = finalize_daily_features(
    daily_stats.sort_index()
)

elapsed = time.perf_counter() - start_time

print("\nFULL AGGREGATION COMPLETE")
print("Records processed:", f"{processed_rows:,}")
print("Daily observations:", len(daily_features))
print("Feature columns:", len(daily_features.columns))
print("Elapsed seconds:", round(elapsed, 2))

# Basic consistency checks
assert processed_rows == 20_343_168
assert daily_features["global_total_events"].sum() == processed_rows
assert list(daily_features.columns) == FEATURE_COLUMNS
assert not daily_features["date"].duplicated().any()

print("\nAll aggregation checks passed.")

display(daily_features.head())


Starting full GDELT feature aggregation...

[1/6] DateAddedMonth=01: 3,667,945 rows | Total: 3,667,945 | Elapsed: 10.8s
[2/6] DateAddedMonth=02: 3,536,167 rows | Total: 7,204,112 | Elapsed: 20.9s
[3/6] DateAddedMonth=03: 3,896,037 rows | Total: 11,100,149 | Elapsed: 31.8s
[4/6] DateAddedMonth=04: 3,780,825 rows | Total: 14,880,974 | Elapsed: 42.6s
[5/6] DateAddedMonth=05: 3,867,054 rows | Total: 18,748,028 | Elapsed: 53.7s
[6/6] DateAddedMonth=06: 1,595,140 rows | Total: 20,343,168 | Elapsed: 58.3s

FULL AGGREGATION COMPLETE
Records processed: 20,343,168
Daily observations: 164
Feature columns: 84
Elapsed seconds: 58.32

All aggregation checks passed.


,date,global_total_events,exporter_total_events,importer_total_events,oil_country_event_share,global_root_01,global_root_02,global_root_03,global_root_04,global_root_05,...,actor1_country_missing_share,actor2_country_missing_share,iran_total_events,saudi_total_events,russia_total_events,iraq_total_events,usa_total_events,china_total_events,iran_goldstein_mean,russia_goldstein_mean
0,2025-01-01,71207.0,6574.0,26239.0,0.450812,9862.0,5069.0,4230.0,16232.0,7422.0,...,0.440490,0.568722,1160.0,459.0,2381.0,230.0,18615.0,2060.0,0.036810,-0.355523
1,2025-01-02,91632.0,7569.0,33571.0,0.438886,12949.0,6389.0,5457.0,20985.0,8556.0,...,0.442236,0.568044,1569.0,868.0,2107.0,336.0,24245.0,2645.0,0.514404,-0.667537
2,2025-01-03,106495.0,7813.0,41148.0,0.450143,14869.0,7303.0,6968.0,25126.0,8874.0,...,0.447148,0.569435,1814.0,560.0,2321.0,360.0,30061.0,2888.0,0.028611,-0.987807
3,2025-01-04,74608.0,5758.0,28572.0,0.451922,10628.0,5098.0,5217.0,18191.0,6114.0,...,0.458449,0.570395,739.0,356.0,1889.0,160.0,20448.0,2202.0,-0.267930,-1.428692
4,2025-01-05,64742.0,5888.0,21070.0,0.408282,8742.0,4327.0,4231.0,16280.0,5202.0,...,0.432022,0.542121,727.0,346.0,2205.0,155.0,14461.0,1564.0,-0.988996,-2.422812



### 22. Validate the Final Daily Feature Dataset

**Objective:** Verify the completeness and consistency of the 84 engineered daily features.

Check date coverage, missing values, infinite values, duplicate dates, and the validity of event counts and proportions.

This validation focuses exclusively on the newly aggregated dataset. The original cleaned event records have already been validated separately.

No existing files will be modified.


In [23]:

# Basic dataset information
print("Dataset shape:", daily_features.shape)
print("Date range:", daily_features["date"].min().date(),
      "to", daily_features["date"].max().date())

print("Unique dates:", daily_features["date"].nunique())
print("Duplicate dates:", daily_features["date"].duplicated().sum())

# Check missing values
missing = daily_features.isna().sum()
missing = missing[missing > 0]

print("\nFeatures containing missing values:")
print(missing if not missing.empty else "None")

# Check infinite numerical values
numeric = daily_features.select_dtypes(include="number")

infinite_count = np.isinf(numeric.to_numpy()).sum()
print("\nInfinite numerical values:", infinite_count)

# Validate event counts
assert (
    daily_features["global_total_events"].sum()
    == 20_343_168
)

assert (
    daily_features["exporter_total_events"]
    <= daily_features["global_total_events"]
).all()

assert (
    daily_features["importer_total_events"]
    <= daily_features["global_total_events"]
).all()

# Validate proportions
SHARE_COLUMNS = [
    col for col in daily_features.columns
    if col.endswith("_share")
]

for col in SHARE_COLUMNS:
    assert daily_features[col].dropna().between(0, 1).all(), (
        f"Invalid proportion detected: {col}"
    )

print("\nShare columns validated:", len(SHARE_COLUMNS))
print("All final feature validation checks passed.")


Dataset shape: (164, 84)
Date range: 2025-01-01 to 2025-06-13
Unique dates: 164
Duplicate dates: 0

Features containing missing values:
None

Infinite numerical values: 0

Share columns validated: 6
All final feature validation checks passed.



### 23. Save the Final Daily Geopolitical Feature Dataset

**Objective:** Export the validated 84-column daily GDELT feature dataset for exploratory analysis and eventual integration with WTI crude oil prices.

Save the dataset in two formats:

- **Parquet:** Efficient storage with preserved data types for analytical processing.
- **CSV:** Convenient for manual inspection and external tools.

The original cleaned event-level dataset remains unchanged. Both outputs contain identical records and feature columns.


In [24]:

# Define output files
PARQUET_OUTPUT = FEATURE_DIR / "gdelt_daily_features.parquet"
CSV_OUTPUT = FEATURE_DIR / "gdelt_daily_features.csv"

# Ensure chronological ordering
daily_features = (
    daily_features
    .sort_values("date")
    .reset_index(drop=True)
)

# Save final dataset
daily_features.to_parquet(
    PARQUET_OUTPUT,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

daily_features.to_csv(
    CSV_OUTPUT,
    index=False
)

# Verify saved files
parquet_check = pd.read_parquet(PARQUET_OUTPUT)
csv_check = pd.read_csv(CSV_OUTPUT, parse_dates=["date"])

assert parquet_check.shape == (164, 84)
assert csv_check.shape == (164, 84)

pd.testing.assert_frame_equal(
    parquet_check,
    daily_features,
    check_dtype=False
)

print("FILES SAVED AND VERIFIED")
print()
print("Parquet:", PARQUET_OUTPUT)
print("CSV:", CSV_OUTPUT)

print("\nDataset shape:", parquet_check.shape)

print("\nFile sizes:")
print(f"Parquet: {PARQUET_OUTPUT.stat().st_size / 1024:.2f} KiB")
print(f"CSV: {CSV_OUTPUT.stat().st_size / 1024:.2f} KiB")

print("\nFinal dataset verification passed.")


FILES SAVED AND VERIFIED

Parquet: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\data\processed\gdelt_daily_features.parquet
CSV: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\data\processed\gdelt_daily_features.csv

Dataset shape: (164, 84)

File sizes:
Parquet: 139.46 KiB
CSV: 146.39 KiB

Final dataset verification passed.



### 24. Document Feature Engineering Methodology

**Objective:** Record the methodology, assumptions, limitations, and important findings from GDELT feature engineering.

#### 1. Dataset and processing

- Input: 20,343,168 cleaned GDELT event records covering January 1–June 13, 2025.
- Processing: Six monthly Parquet partitions, read in batches of 100,000 records.
- Output: 164 daily observations with 84 columns (date + 83 numerical features).
- Processing time: 58.32 seconds.
- Reporting date: `DATEADDED`, rather than `SQLDATE`, to represent when events became available in GDELT.

#### 2. Feature categories

The dataset includes:

- Global geopolitical event activity.
- Worldwide CAMEO event categories.
- Oil exporter/importer CAMEO categories.
- Directed exporter/importer relationships.
- Goldstein geopolitical scores.
- News tone and media attention.
- Internal-event shares and geographic data-quality indicators.
- Country-specific geopolitical indicators.

#### 3. Oil-exporting and oil-importing countries

We retained the historical country groups selected from Vodenska et al. (2016).

A reference-data compatibility issue was discovered: GDELT actor-country fields use three-letter country codes, whereas our initial reference lists used two-letter codes. The reference mappings were corrected before feature extraction.

Events involving either Actor1 or Actor2 from a group contribute to that group's broad activity indicators. An event is counted only once within each group but can belong to both broad exporter and importer groups.

Directed relationship categories use Actor1 as the source and Actor2 as the destination. The four categories are mutually exclusive.

Missing actor-country information is not replaced using geographic event-location fields.

#### 4. Goldstein Impact Index

Following the definition used in Vodenska et al. (2016):

Goldstein Impact Index = Σ(GoldsteinScale_Adjusted × NumMentions) / Number of valid events

This is not a conventional weighted mean, because the denominator is the number of events rather than the sum of mentions.

Consequently, the index is not restricted to the original Goldstein scale of −10 to +10.

In our 10,000-record validation sample, the exporter Goldstein Impact Index was −39.9187. This value was retained rather than clipped.

We will investigate the index's distribution and extreme values during EDA.

#### 5. Media-attention indicators

`NumMentions` and `NumArticles` are aggregated across GDELT event records.

These totals measure event-associated reporting intensity. They do not necessarily represent distinct news articles, because one article can be associated with multiple events.

#### 6. Internal geopolitical events

For this implementation, an internal event is defined as an event where both actors have the same nonmissing country code.

This is an operational actor-based definition and should not be confused with a geographic definition based on the event's physical location.

#### 7. Missing geographic information

Actor-country missingness is preserved and quantified through two daily features.

For the 10,000-record validation sample:

- Actor1 country missing: 46.04%.
- Actor2 country missing: 60.14%.

Missing actor-country information limits the number of events that can be classified into directed country-group relationships.

#### 8. Feature normalization

The original interpretable features are preserved.

During EDA, we will investigate:

- Event counts expressed as shares of appropriate daily totals.
- `log1p` transformations for heavily skewed nonnegative counts.
- An `asinh` transformation for the signed Goldstein Impact Index.
- Distributional differences between daily indicators.

Normalization will not automatically replace the original features. Any machine-learning scaler must be fitted only on training data.

#### 9. Validation and reproducibility

Feature calculations were first tested on 10,000 records.

The scalable batch aggregation reproduced all 84 sample features using an independent calculation method.

The full dataset passed validation:

- 20,343,168 records accounted for.
- 164 unique dates.
- 84 columns.
- No missing or infinite feature values.
- No duplicate dates.
- All six share features within [0, 1].

Final outputs were saved as:

- `data/processed/gdelt_daily_features.parquet`
- `data/processed/gdelt_daily_features.csv`

#### 10. Forecasting considerations

The dataset contains daily geopolitical indicators, not yet a forecasting model.

Future WTI integration must account for market trading days, reporting availability, and potential information leakage.

Lagged geopolitical features will be investigated, including the two-day lag suggested by findings in Elshendy et al. for GDELT-based media activity. That study used GDELT GKG data, which differs from our GDELT event data.

Time-series validation must preserve chronological order.



### 25. Create the Feature Engineering Module

**Objective:** Create a reusable Python module for the validated GDELT daily feature-engineering pipeline.

We will move the feature definitions, country classifications, batch aggregation logic, and final daily feature calculations from the notebook into `src/feature_engineering.py`.

The module will eventually:

1. Read cleaned GDELT Parquet partitions in batches.
2. Calculate daily intermediate statistics.
3. Combine statistics across all partitions.
4. Generate the same 84 daily geopolitical features.
5. Save the resulting dataset in Parquet and CSV formats.

The original cleaned data will remain unchanged.

We will first create the module file, then transfer and validate the functions incrementally.


In [25]:

# Cell 25 — Create the feature engineering module

from pathlib import Path

SRC_DIR = PROJECT_DIR / "src"
FEATURE_MODULE = SRC_DIR / "feature_engineering.py"

# Ensure src directory exists
SRC_DIR.mkdir(parents=True, exist_ok=True)

# Do not overwrite an existing module
if FEATURE_MODULE.exists():
    print("Module already exists:", FEATURE_MODULE)
    print("Existing file preserved.")
else:
    FEATURE_MODULE.write_text(
        '"""GDELT daily geopolitical feature engineering.\n\n'
        'Converts cleaned event-level Parquet data into daily features\n'
        'for crude oil price analysis and forecasting.\n'
        '"""\n',
        encoding="utf-8"
    )

    print("Module created successfully.")

print("\nFile:", FEATURE_MODULE)
print("Size:", FEATURE_MODULE.stat().st_size, "bytes")
print("\nCurrent contents:")
print(FEATURE_MODULE.read_text(encoding="utf-8"))


Module created successfully.

File: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\src\feature_engineering.py
Size: 167 bytes

Current contents:
"""GDELT daily geopolitical feature engineering.

Converts cleaned event-level Parquet data into daily features
for crude oil price analysis and forecasting.
"""




### 26. Add Reference Data and Feature Definitions

**Objective:** Transfer the reference definitions used in the notebook into the reusable feature-engineering module.

This step includes the oil-exporting and oil-importing country mappings, CAMEO root categories, selected major countries, and the 84-column feature specification.

These definitions ensure that the standalone module uses the same classifications and feature names as the validated notebook.

The existing module is preserved if the expected reference definitions are not available.


In [26]:

# Cell 26 — Transfer validated feature definitions

import pprint

# Verify notebook definitions exist
required_names = [
    "OIL_EXPORTERS", "OIL_IMPORTERS",
    "CAMEO_ROOTS", "FEATURE_GROUPS",
    "MAJOR_COUNTRIES", "REQUIRED_COLUMNS"
]

missing_names = [
    name for name in required_names
    if name not in globals()
]

assert not missing_names, (
    f"Missing notebook definitions: {missing_names}"
)

definitions = {
    "OIL_EXPORTERS": OIL_EXPORTERS,
    "OIL_IMPORTERS": OIL_IMPORTERS,
    "CAMEO_ROOTS": CAMEO_ROOTS,
    "FEATURE_GROUPS": FEATURE_GROUPS,
    "MAJOR_COUNTRIES": MAJOR_COUNTRIES,
    "REQUIRED_COLUMNS": REQUIRED_COLUMNS,
}

# Create standalone Python definitions
source = (
    '"""GDELT daily geopolitical feature engineering."""\n\n'
    "from pathlib import Path\n"
    "import numpy as np\n"
    "import pandas as pd\n"
    "import pyarrow.parquet as pq\n\n"
)

for name, value in definitions.items():
    source += (
        f"{name} = "
        f"{pprint.pformat(value, sort_dicts=False, width=88)}\n\n"
    )

source += """
EXPORTER_CODES = set(OIL_EXPORTERS)
IMPORTER_CODES = set(OIL_IMPORTERS)
OIL_COUNTRIES = EXPORTER_CODES | IMPORTER_CODES

FEATURE_COLUMNS = [
    name
    for features in FEATURE_GROUPS.values()
    for name in features
]

RELATIONSHIPS = [
    "exp_to_exp",
    "exp_to_imp",
    "imp_to_exp",
    "imp_to_imp",
]
"""

# Prevent accidental replacement if functions already exist
existing = FEATURE_MODULE.read_text(encoding="utf-8")

assert "def aggregate_feature_batch" not in existing, (
    "Existing module contains aggregation functions. "
    "Automatic replacement stopped."
)

# Write reference definitions
FEATURE_MODULE.write_text(source, encoding="utf-8")

# Validate syntax without executing the module
compile(source, str(FEATURE_MODULE), "exec")

print("Reference definitions written successfully.")
print("Module:", FEATURE_MODULE)
print("File size:", FEATURE_MODULE.stat().st_size, "bytes")
print("Feature columns:", len(FEATURE_COLUMNS))
print("Country mappings:", len(OIL_EXPORTERS) + len(OIL_IMPORTERS))
print("CAMEO categories:", len(CAMEO_ROOTS))
print("Python syntax validation: PASSED")


Reference definitions written successfully.
Module: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\src\feature_engineering.py
File size: 5504 bytes
Feature columns: 84
Country mappings: 20
CAMEO categories: 20
Python syntax validation: PASSED



### 27. Transfer the Batch Aggregation Function

**Objective:** Add the validated batch-processing function to the reusable feature-engineering module.

We transfer `aggregate_feature_batch()` from the notebook into `src/feature_engineering.py` without changing its calculations.

The function calculates 100 intermediate statistics per reporting date, including event counts, CAMEO classifications, Goldstein scores, news tone, media attention, and country-level indicators.

We then load the function from the standalone module and compare its output with the notebook's previously validated results.

This confirms that moving the function into a separate Python file does not change its behavior.


In [27]:

# Cell 27 — Transfer and validate batch aggregation

import inspect
import importlib.util
import textwrap

# Retrieve the existing validated function from the notebook
function_source = textwrap.dedent(
    inspect.getsource(aggregate_feature_batch)
)

# Read the existing module
module_source = FEATURE_MODULE.read_text(encoding="utf-8")

# Avoid accidentally adding the function twice
assert "def aggregate_feature_batch(" not in module_source, (
    "Aggregation function already exists in the module."
)

# Append the validated function
updated_source = (
    module_source.rstrip()
    + "\n\n\n"
    + function_source
    + "\n"
)

# Check Python syntax before writing
compile(updated_source, str(FEATURE_MODULE), "exec")

FEATURE_MODULE.write_text(updated_source, encoding="utf-8")

# Import the standalone module directly
spec = importlib.util.spec_from_file_location(
    "gdelt_feature_engineering",
    FEATURE_MODULE
)

feature_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(feature_module)

# Run the standalone function against the original sample
module_batch_test = feature_module.aggregate_feature_batch(sample)

# Compare with the validated notebook result from Cell 19
pd.testing.assert_frame_equal(
    module_batch_test,
    batch_test,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)

print("Aggregation function transferred successfully.")
print("Module:", FEATURE_MODULE)
print("Intermediate statistics:", len(module_batch_test.columns))
print("Sample records:", len(sample))
print("Module vs notebook comparison: PASSED")
print("Python syntax validation: PASSED")


Aggregation function transferred successfully.
Module: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\src\feature_engineering.py
Intermediate statistics: 100
Sample records: 10000
Module vs notebook comparison: PASSED
Python syntax validation: PASSED



### 28. Transfer the Daily Feature Finalization Function

**Objective:** Add the validated `finalize_daily_features()` function to the standalone feature-engineering module.

The function converts additive daily statistics into the final 84-column dataset, including Goldstein means, standard deviations, impact indices, news tone, event shares, and country-level indicators.

We transfer the function without changing its calculations, then compare the standalone module's output against the validated notebook results.

Successful validation confirms that both core feature-engineering functions can operate independently of the notebook.


In [28]:

# Cell 28 — Transfer and validate feature finalization

import inspect
import importlib.util
import textwrap

# Retrieve the validated notebook function
function_source = textwrap.dedent(
    inspect.getsource(finalize_daily_features)
)

# Read the existing standalone module
module_source = FEATURE_MODULE.read_text(encoding="utf-8")

assert "def finalize_daily_features(" not in module_source, (
    "Finalization function already exists in the module."
)

# Append the function and validate Python syntax
updated_source = (
    module_source.rstrip()
    + "\n\n\n"
    + function_source
    + "\n"
)

compile(updated_source, str(FEATURE_MODULE), "exec")
FEATURE_MODULE.write_text(updated_source, encoding="utf-8")

# Reload module to include the new function
spec = importlib.util.spec_from_file_location(
    "gdelt_feature_engineering",
    FEATURE_MODULE
)

feature_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(feature_module)

# Test both standalone functions together
module_batch_test = feature_module.aggregate_feature_batch(sample)

module_final_test = feature_module.finalize_daily_features(
    module_batch_test
)

# Compare against the independently validated 84-feature sample
pd.testing.assert_frame_equal(
    module_final_test,
    daily_features_test,
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

print("Finalization function transferred successfully.")
print("Module:", FEATURE_MODULE)
print("Final dataset shape:", module_final_test.shape)
print("Module vs notebook comparison: PASSED")
print("Python syntax validation: PASSED")
print("\nBoth core functions operate independently of the notebook.")


Finalization function transferred successfully.
Module: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\src\feature_engineering.py
Final dataset shape: (1, 84)
Module vs notebook comparison: PASSED
Python syntax validation: PASSED

Both core functions operate independently of the notebook.



### 29. Create the Standalone Feature Engineering Pipeline

**Objective:** Add a reusable function that performs the complete GDELT feature-engineering workflow without relying on notebook variables.

The pipeline will:

1. Discover cleaned Parquet files in the input directory.
2. Read event records in batches of 100,000.
3. Aggregate intermediate daily statistics across all batches.
4. Generate the final 84 daily features.
5. Validate the resulting dataset.
6. Export the dataset to Parquet and CSV.

The function accepts configurable input and output directories, allowing it to be reused with future GDELT datasets.

The existing cleaned event files will remain unchanged.

This cell defines and transfers the pipeline function. It does not execute the complete 20-million-record processing job yet.


In [29]:

# Cell 29 — Add standalone pipeline function

import inspect
import importlib.util
import textwrap

def run_feature_pipeline(input_dir, output_dir, batch_size=100_000):
    """Build daily GDELT geopolitical features from cleaned Parquet files."""

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    parquet_files = sorted(input_dir.rglob("*.parquet"))

    if not parquet_files:
        raise FileNotFoundError(
            f"No Parquet files found in {input_dir}"
        )

    if batch_size < 1:
        raise ValueError("batch_size must be a positive integer")

    daily_stats = None
    processed_rows = 0

    for file_number, file in enumerate(parquet_files, start=1):
        parquet = pq.ParquetFile(file)
        file_rows = 0

        for batch in parquet.iter_batches(
            batch_size=batch_size,
            columns=REQUIRED_COLUMNS
        ):
            df = batch.to_pandas()

            batch_stats = aggregate_feature_batch(df)

            if daily_stats is None:
                daily_stats = batch_stats
            else:
                daily_stats = daily_stats.add(
                    batch_stats,
                    fill_value=0
                )

            file_rows += len(df)
            processed_rows += len(df)

            del df, batch_stats

        print(
            f"[{file_number}/{len(parquet_files)}] "
            f"{file.parent.name}: {file_rows:,} records",
            flush=True
        )

    daily_features = finalize_daily_features(
        daily_stats.sort_index()
    )

    # Validate resulting dataset
    if daily_features["global_total_events"].sum() != processed_rows:
        raise AssertionError("Event counts do not match input records")

    if list(daily_features.columns) != FEATURE_COLUMNS:
        raise AssertionError("Feature columns do not match specification")

    if daily_features["date"].duplicated().any():
        raise AssertionError("Duplicate reporting dates detected")

    if not np.isfinite(
        daily_features.select_dtypes(include="number").to_numpy()
    ).all():
        raise AssertionError("Non-finite feature values detected")

    daily_features = (
        daily_features
        .sort_values("date")
        .reset_index(drop=True)
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    parquet_output = output_dir / "gdelt_daily_features.parquet"
    csv_output = output_dir / "gdelt_daily_features.csv"

    daily_features.to_parquet(
        parquet_output,
        index=False,
        engine="pyarrow",
        compression="snappy"
    )

    daily_features.to_csv(csv_output, index=False)

    print("\nFEATURE ENGINEERING COMPLETE")
    print("Records processed:", f"{processed_rows:,}")
    print("Daily observations:", len(daily_features))
    print("Feature columns:", len(daily_features.columns))
    print("Parquet:", parquet_output)
    print("CSV:", csv_output)

    return daily_features


# Transfer the function to the standalone module
function_source = textwrap.dedent(
    inspect.getsource(run_feature_pipeline)
)

module_source = FEATURE_MODULE.read_text(encoding="utf-8")

assert "def run_feature_pipeline(" not in module_source, (
    "Pipeline function already exists in the module."
)

updated_source = (
    module_source.rstrip()
    + "\n\n\n"
    + function_source
    + "\n"
)

compile(updated_source, str(FEATURE_MODULE), "exec")
FEATURE_MODULE.write_text(updated_source, encoding="utf-8")

# Reload and verify module
spec = importlib.util.spec_from_file_location(
    "gdelt_feature_engineering",
    FEATURE_MODULE
)

feature_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(feature_module)

assert callable(feature_module.run_feature_pipeline)

print("Standalone feature pipeline function added successfully.")
print("Module:", FEATURE_MODULE)
print("Function available:", hasattr(feature_module, "run_feature_pipeline"))
print("Python syntax validation: PASSED")


Standalone feature pipeline function added successfully.
Module: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\src\feature_engineering.py
Function available: True
Python syntax validation: PASSED



### 30. Validate Standalone Feature Engineering Reproducibility

**Objective:** Verify that the standalone Python module reproduces the previously validated daily GDELT feature dataset.

We will process all six cleaned monthly Parquet partitions using `run_feature_pipeline()` and compare the resulting dataset against the 164 × 84 reference dataset generated earlier.

The pipeline will write its outputs to a separate test directory to avoid overwriting the validated reference files.

Validation will include dataset dimensions, dates, feature columns, numerical values, and total event counts.

Successful verification demonstrates that the feature-engineering workflow can be reproduced without relying on notebook calculations.


In [30]:

# Cell 30 — End-to-end reproducibility test

import time

# Separate output directory to protect validated files
TEST_OUTPUT_DIR = PROJECT_DIR / "data" / "processed" / "pipeline_test"

# Load the previously validated reference dataset
reference_file = FEATURE_DIR / "gdelt_daily_features.parquet"
reference_df = pd.read_parquet(reference_file)

# Run standalone pipeline on all cleaned Parquet files
start_time = time.perf_counter()

pipeline_df = feature_module.run_feature_pipeline(
    input_dir=CLEANED_DIR,
    output_dir=TEST_OUTPUT_DIR,
    batch_size=100_000
)

elapsed = time.perf_counter() - start_time

# Ensure both datasets have the same row and column order
reference_df = reference_df.sort_values("date").reset_index(drop=True)
pipeline_df = pipeline_df.sort_values("date").reset_index(drop=True)

assert pipeline_df.shape == reference_df.shape
assert list(pipeline_df.columns) == list(reference_df.columns)

# Compare all 84 features and all 164 dates
pd.testing.assert_frame_equal(
    pipeline_df,
    reference_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

assert pipeline_df["global_total_events"].sum() == 20_343_168

print("\nEND-TO-END REPRODUCIBILITY TEST")
print("Reference shape:", reference_df.shape)
print("Pipeline shape:", pipeline_df.shape)
print("Dates match:", pipeline_df["date"].equals(reference_df["date"]))
print("All feature values match: YES")
print("Records accounted for:", f"{int(pipeline_df['global_total_events'].sum()):,}")
print("Processing time:", round(elapsed, 2), "seconds")
print("\nREPRODUCIBILITY TEST PASSED")


[1/6] DateAddedMonth=01: 3,667,945 records
[2/6] DateAddedMonth=02: 3,536,167 records
[3/6] DateAddedMonth=03: 3,896,037 records
[4/6] DateAddedMonth=04: 3,780,825 records
[5/6] DateAddedMonth=05: 3,867,054 records
[6/6] DateAddedMonth=06: 1,595,140 records

FEATURE ENGINEERING COMPLETE
Records processed: 20,343,168
Daily observations: 164
Feature columns: 84
Parquet: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\data\processed\pipeline_test\gdelt_daily_features.parquet
CSV: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\data\processed\pipeline_test\gdelt_daily_features.csv

END-TO-END REPRODUCIBILITY TEST
Reference shape: (164, 84)
Pipeline shape: (164, 84)
Dates match: True
All feature values match: YES
Records accounted for: 20,343,168
Processing time: 57.16 seconds

REPRODUCIBILITY TEST PASSED



### 31. Inspect the Existing Pipeline Before Integration

**Objective:** Examine the existing `src/pipeline.py` to determine how to integrate the feature-engineering stage without disrupting the validated cleaning pipeline.

The cleaning pipeline already supports processing GDELT records and a `--test` execution mode.

Before modifying the code, we inspect its imports, functions, command-line arguments, and execution structure.

This ensures that feature engineering can be added as a separate pipeline stage while preserving existing cleaning functionality.

This step is read-only and does not modify any project files.


In [31]:

# Cell 31 — Inspect the existing cleaning pipeline

PIPELINE_FILE = PROJECT_DIR / "src" / "pipeline.py"

assert PIPELINE_FILE.exists(), (
    f"Pipeline file not found: {PIPELINE_FILE}"
)

source = PIPELINE_FILE.read_text(encoding="utf-8")
lines = source.splitlines()

print("Pipeline:", PIPELINE_FILE)
print("Total lines:", len(lines))

print("\nIMPORTS, FUNCTIONS, AND ENTRY POINTS")
print("-" * 50)

for number, line in enumerate(lines, start=1):
    stripped = line.strip()

    if (
        stripped.startswith(("import ", "from ", "def ", "class "))
        or "ArgumentParser(" in stripped
        or "add_argument(" in stripped
        or '__name__ == "__main__"' in stripped
        or "__name__ == '__main__'" in stripped
    ):
        print(f"{number:4}: {line}")

print("\nLAST 60 LINES")
print("-" * 50)

start = max(0, len(lines) - 60)

for number in range(start, len(lines)):
    print(f"{number + 1:4}: {lines[number]}")


Pipeline: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\src\pipeline.py
Total lines: 246

IMPORTS, FUNCTIONS, AND ENTRY POINTS
--------------------------------------------------
   1: from pathlib import Path
   2: import argparse
   3: import time
   5: import pandas as pd
   6: import pyarrow as pa
   7: import pyarrow.parquet as pq
   9: from data_cleaning import clean_gdelt
  33: def load_references():
  65: def get_dtype_map():
  91: def run_pipeline(test_mode=False):
 236: if __name__ == "__main__":
 237:     parser = argparse.ArgumentParser()
 239:     parser.add_argument(

LAST 60 LINES
--------------------------------------------------
 187:                     / f"DateAddedMonth={month}"
 188:                 )
 189: 
 190:                 partition_dir.mkdir(
 191:                     parents=True,
 192:                     exist_ok=True
 193:                 )
 194: 
 195:                 if key not in writers:
 196:     


### 32. Inspect Pipeline Command-Line Execution

**Objective:** Identify the exact code responsible for launching the existing cleaning pipeline.

We will inspect the final section of `src/pipeline.py`, including its argument parser and execution logic.

This allows us to introduce a separate feature-engineering mode while preserving the current cleaning behavior and `--test` option.

The inspection is read-only. No files will be modified.


In [32]:

# Cell 32 — Inspect the pipeline entry point

PIPELINE_FILE = PROJECT_DIR / "src" / "pipeline.py"

lines = PIPELINE_FILE.read_text(encoding="utf-8").splitlines()

print("PIPELINE ENTRY POINT")
print("-" * 60)

# Display the final 30 lines
start = max(0, len(lines) - 30)

for i in range(start, len(lines)):
    print(f"{i + 1:4}: {lines[i]}")


PIPELINE ENTRY POINT
------------------------------------------------------------
 217:                     f"Rows: {total_rows:,}"
 218:                 )
 219: 
 220:     finally:
 221:         for writer in writers.values():
 222:             writer.close()
 223: 
 224:     (OUTPUT_DIR / "_SUCCESS").touch()
 225: 
 226:     elapsed = (time.time() - start_time) / 60
 227: 
 228:     print("\nPipeline completed.")
 229:     print("Total chunks:", total_chunks)
 230:     print("Total rows:", f"{total_rows:,}")
 231:     print("Partitions:", len(writers))
 232:     print("Processing time (minutes):", round(elapsed, 2))
 233:     print("Output:", OUTPUT_DIR)
 234: 
 235: 
 236: if __name__ == "__main__":
 237:     parser = argparse.ArgumentParser()
 238: 
 239:     parser.add_argument(
 240:         "--test",
 241:         action="store_true",
 242:         help="Test cleaning on 100,000 records without saving files"
 243:     )
 244: 
 245:     args = parser.parse_args()
 246:     run_p


### 33. Integrate Feature Engineering into the Main Pipeline

**Objective:** Add a separate feature-engineering execution mode to `src/pipeline.py`.

The existing cleaning pipeline will remain unchanged. A new `--features` argument will execute the validated standalone feature-engineering module using the cleaned Parquet dataset.

Supported commands:

- `python pipeline.py` — Run the existing cleaning pipeline.
- `python pipeline.py --test` — Run the existing cleaning test.
- `python pipeline.py --features` — Generate daily geopolitical features.

We will back up the original pipeline before modifying its command-line entry point.

This cell modifies the pipeline script but does not execute data processing.


In [33]:

# Cell 33 — Add feature-engineering CLI mode

from datetime import datetime
import ast

PIPELINE_FILE = PROJECT_DIR / "src" / "pipeline.py"

source = PIPELINE_FILE.read_text(encoding="utf-8")

# Find the existing command-line entry point
entry_marker = 'if __name__ == "__main__":'

assert source.count(entry_marker) == 1, (
    "Expected exactly one pipeline entry point."
)

# Preserve the original import and cleaning functions
prefix = source.split(entry_marker)[0]

# Prevent accidentally adding this integration twice
assert "from feature_engineering import run_feature_pipeline" not in source, (
    "Feature engineering integration already exists."
)

# Back up the original script
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_file = PIPELINE_FILE.with_name(
    f"pipeline_backup_{timestamp}.py"
)

backup_file.write_text(source, encoding="utf-8")

# Add the feature-engineering import
prefix = (
    prefix.rstrip()
    + "\n\n"
    + "from feature_engineering import run_feature_pipeline\n\n\n"
)

# New command-line entry point
new_entry = '''
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="GDELT cleaning and feature engineering pipeline"
    )

    parser.add_argument(
        "--test",
        action="store_true",
        help="Run the existing cleaning pipeline in test mode"
    )

    parser.add_argument(
        "--features",
        action="store_true",
        help="Run daily geopolitical feature engineering"
    )

    args = parser.parse_args()

    if args.features:
        if args.test:
            parser.error("--test and --features cannot be used together")

        project_dir = Path(__file__).resolve().parent.parent

        cleaned_dir = (
            project_dir
            / "gdelt_data"
            / "processed"
            / "gdelt_cleaned_by_date_v2"
        )

        feature_output_dir = (
            project_dir / "data" / "processed"
        )

        run_feature_pipeline(
            input_dir=cleaned_dir,
            output_dir=feature_output_dir,
            batch_size=100_000
        )
    else:
        run_pipeline(test_mode=args.test)
'''

# Construct and validate updated script
updated_source = prefix + new_entry

ast.parse(updated_source)

PIPELINE_FILE.write_text(updated_source, encoding="utf-8")

print("Pipeline integration completed.")
print("Backup:", backup_file)
print("Updated:", PIPELINE_FILE)
print("Feature engineering option: --features")
print("Cleaning test option: --test")
print("Python syntax validation: PASSED")


Pipeline integration completed.
Backup: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\src\pipeline_backup_20261022_101526.py
Updated: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\src\pipeline.py
Feature engineering option: --features
Cleaning test option: --test
Python syntax validation: PASSED



### 34. Test the Integrated Pipeline Command-Line Interface

**Objective:** Verify that the updated `pipeline.py` correctly supports cleaning and feature engineering without disrupting existing functionality.

We will perform three checks:

1. Run `--help` to confirm the available command-line options.
2. Run `--test` to verify that the existing cleaning test still works.
3. Run `--features --test` to confirm that incompatible execution modes are rejected.

These checks do not execute the full cleaning or feature-engineering pipelines.

The existing datasets remain unchanged.


In [34]:

# Cell 34 — Test integrated pipeline command-line options

import subprocess
import sys

PIPELINE_FILE = PROJECT_DIR / "src" / "pipeline.py"

def run_cli_check(arguments):
    command = [sys.executable, str(PIPELINE_FILE), *arguments]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        cwd=str(PROJECT_DIR)
    )

    return result

# 1. Verify --help
help_result = run_cli_check(["--help"])

print("TEST 1 — Command-line help")
print(help_result.stdout)

assert help_result.returncode == 0
assert "--features" in help_result.stdout
assert "--test" in help_result.stdout

# 2. Verify existing cleaning test
cleaning_result = run_cli_check(["--test"])

print("\nTEST 2 — Existing cleaning test")
print(cleaning_result.stdout[-1500:])

if cleaning_result.returncode != 0:
    print(cleaning_result.stderr[-1500:])

assert cleaning_result.returncode == 0, (
    "Existing cleaning test failed."
)

# 3. Verify incompatible arguments are rejected
conflict_result = run_cli_check(["--features", "--test"])

print("\nTEST 3 — Incompatible arguments")
print(conflict_result.stderr[-500:])

assert conflict_result.returncode != 0
assert "cannot be used together" in conflict_result.stderr

print("\nALL COMMAND-LINE TESTS PASSED")


TEST 1 — Command-line help
usage: pipeline.py [-h] [--test] [--features]

GDELT cleaning and feature engineering pipeline

options:
  -h, --help  show this help message and exit
  --test      Run the existing cleaning pipeline in test mode
  --features  Run daily geopolitical feature engineering


TEST 2 — Existing cleaning test
Loading reference files...
Test rows: 100000
Test columns: 90
Quality flags: 28


TEST 3 — Incompatible arguments
usage: pipeline.py [-h] [--test] [--features]
pipeline.py: error: --test and --features cannot be used together


ALL COMMAND-LINE TESTS PASSED



### 35. Validate Feature Engineering Through the Main Pipeline

**Objective:** Verify that the integrated `src/pipeline.py` can execute feature engineering independently of the Jupyter notebook.

We will run the pipeline using the `--features` command-line argument.

The pipeline will process all six cleaned monthly Parquet partitions and regenerate the 164-day, 84-column geopolitical feature dataset.

To verify reproducibility, we load the existing validated dataset before execution and compare it against the regenerated output afterward.

This confirms that the complete feature-engineering workflow is accessible through the project's main command-line pipeline.

The command will overwrite the two existing feature-output files, but only after the reference dataset has been loaded into memory for comparison. The cleaned event-level data remains unchanged.


In [35]:

# Cell 35 — Full command-line integration test

import subprocess
import sys
import time

PIPELINE_FILE = PROJECT_DIR / "src" / "pipeline.py"
PARQUET_OUTPUT = FEATURE_DIR / "gdelt_daily_features.parquet"
CSV_OUTPUT = FEATURE_DIR / "gdelt_daily_features.csv"

# Load validated reference before pipeline overwrites output files
reference_df = pd.read_parquet(PARQUET_OUTPUT)

print("Running integrated feature-engineering pipeline...\n")

start_time = time.perf_counter()

result = subprocess.run(
    [sys.executable, str(PIPELINE_FILE), "--features"],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_DIR)
)

elapsed = time.perf_counter() - start_time

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Feature-engineering pipeline failed.")

# Load regenerated dataset
regenerated_df = pd.read_parquet(PARQUET_OUTPUT)

# Compare reference and regenerated outputs
pd.testing.assert_frame_equal(
    regenerated_df,
    reference_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

assert CSV_OUTPUT.exists()
assert regenerated_df.shape == (164, 84)
assert regenerated_df["global_total_events"].sum() == 20_343_168

print("\nFINAL PIPELINE INTEGRATION TEST")
print("Pipeline exit code:", result.returncode)
print("Dataset shape:", regenerated_df.shape)
print("All 84 features match reference: YES")
print("Original events accounted for: 20,343,168")
print("Execution time:", round(elapsed, 2), "seconds")
print("\nFULL PIPELINE INTEGRATION PASSED")


Running integrated feature-engineering pipeline...

[1/6] DateAddedMonth=01: 3,667,945 records
[2/6] DateAddedMonth=02: 3,536,167 records
[3/6] DateAddedMonth=03: 3,896,037 records
[4/6] DateAddedMonth=04: 3,780,825 records
[5/6] DateAddedMonth=05: 3,867,054 records
[6/6] DateAddedMonth=06: 1,595,140 records

FEATURE ENGINEERING COMPLETE
Records processed: 20,343,168
Daily observations: 164
Feature columns: 84
Parquet: C:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\data\processed\gdelt_daily_features.parquet
CSV: C:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\data\processed\gdelt_daily_features.csv


FINAL PIPELINE INTEGRATION TEST
Pipeline exit code: 0
Dataset shape: (164, 84)
All 84 features match reference: YES
Original events accounted for: 20,343,168
Execution time: 57.04 seconds

FULL PIPELINE INTEGRATION PASSED



### 36. Integrate Brent Crude Oil Prices with GDELT

**Objective:** Merge daily Brent crude oil prices with our 84-column geopolitical feature dataset.

The Brent dataset contains daily crude oil prices in USD per barrel. Missing observations occur on non-reporting days, including weekends and holidays.

We preserve all 164 GDELT calendar days and retain missing Brent observations rather than inventing prices.

Daily Brent returns are calculated from consecutive valid price observations, and their dates are preserved for subsequent forecasting analysis.


In [37]:

# Cell 36 — Load and merge Brent prices

BRENT_FILE = PROJECT_DIR / "data" / "raw" / "DCOILBRENTEU.csv"

brent = pd.read_csv(BRENT_FILE)

# Support common FRED CSV date headings
date_col = (
    "observation_date"
    if "observation_date" in brent.columns
    else "DATE"
)

brent = brent.rename(columns={
    date_col: "date",
    "DCOILBRENTEU": "brent_price"
})

brent["date"] = pd.to_datetime(brent["date"])

brent["brent_price"] = pd.to_numeric(
    brent["brent_price"], errors="coerce"
)

brent = brent.sort_values("date").reset_index(drop=True)

# Calculate returns between consecutive observed prices
observed = brent.dropna(subset=["brent_price"]).copy()
observed["brent_return"] = (
    observed["brent_price"].pct_change(fill_method=None)
)

# Merge while preserving every GDELT reporting day
merged = daily_features.merge(
    observed[["date", "brent_price", "brent_return"]],
    on="date",
    how="left",
    validate="one_to_one"
)

print("GDELT observations:", len(daily_features))
print("Merged observations:", len(merged))
print("Merged columns:", len(merged.columns))
print("Days with Brent prices:", merged["brent_price"].notna().sum())
print("Days without Brent prices:", merged["brent_price"].isna().sum())

assert len(merged) == 164
assert not merged["date"].duplicated().any()

display(
    merged[["date", "global_total_events",
            "brent_price", "brent_return"]].head(10)
)


GDELT observations: 164
Merged observations: 164
Merged columns: 86
Days with Brent prices: 113
Days without Brent prices: 51


,date,global_total_events,brent_price,brent_return
0,2025-01-01,71207.0,NaN,NaN
1,2025-01-02,91632.0,76.14,NaN
2,2025-01-03,106495.0,76.72,0.007618
3,2025-01-04,74608.0,NaN,NaN
4,2025-01-05,64742.0,NaN,NaN
5,2025-01-06,110445.0,77.27,0.007169
6,2025-01-07,129240.0,77.84,0.007377
7,2025-01-08,136893.0,77.37,-0.006038
8,2025-01-09,140097.0,78.44,0.013830
9,2025-01-10,129292.0,79.76,0.016828



### 37. Validate and Save the GDELT–Brent Dataset

**Objective:** Validate the combined geopolitical and Brent crude oil dataset and save it for exploratory analysis.

We preserve all original GDELT feature columns and add Brent price and return indicators.

Missing Brent prices are retained. We do not forward-fill, interpolate, or assume that prices were observed on non-trading days.

The resulting dataset will support lagged geopolitical analysis and subsequent predictive modeling.


In [38]:

# Cell 37 — Validate and save merged dataset

assert len(merged) == len(daily_features)
assert list(merged.columns[:84]) == FEATURE_COLUMNS

assert (
    merged["global_total_events"].sum() == 20_343_168
)

assert (merged["brent_price"].dropna() > 0).all()

OUTPUT_PARQUET = FEATURE_DIR / "gdelt_brent_daily.parquet"
OUTPUT_CSV = FEATURE_DIR / "gdelt_brent_daily.csv"

merged.to_parquet(OUTPUT_PARQUET, index=False)
merged.to_csv(OUTPUT_CSV, index=False)

# Verify exported Parquet
saved = pd.read_parquet(OUTPUT_PARQUET)

pd.testing.assert_frame_equal(saved, merged)

print("GDELT–BRENT INTEGRATION COMPLETE")
print("Rows:", len(saved))
print("Columns:", len(saved.columns))
print("Brent observations:", saved.brent_price.notna().sum())
print("Missing Brent prices:", saved.brent_price.isna().sum())
print("\nParquet:", OUTPUT_PARQUET)
print("CSV:", OUTPUT_CSV)
print("\nValidation passed.")


GDELT–BRENT INTEGRATION COMPLETE
Rows: 164
Columns: 86
Brent observations: 113
Missing Brent prices: 51

Parquet: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\data\processed\gdelt_brent_daily.parquet
CSV: c:\Users\kevin\school\Fall 2026\Data Intensive Computing\DIC Semeters Project\Phase 2\Data_wizards\data\processed\gdelt_brent_daily.csv

Validation passed.
